In [56]:
import pandas as pd
import re
import spacy
import simplemma
from tqdm import tqdm

In [57]:
df = pd.read_csv("guardian_environment_news.csv")
df.head()

,Title,Intro Text,Authors,Article Text,Date Published
0,Liz Truss ‘will approve more oil drilling if ...,Tory leadership candidate criticised by campai...,"['Rob Davies', '@ByRobDavies']",Liz Truss will sign off on a push for more oil...,2022-08-30
1,Renewed Highland golf course plan has environm...,Scottish government rejected a new links at Co...,"['Ewan Murray', '@mrewanmurray']",It is an area so tranquil that the notion of b...,2021-03-22
2,Visiting green spaces deters mental health dr...,Positive effects were stronger among those rep...,"['Damien Gayle', '@damiengayle']","Visits to parks, community gardens and other u...",2023-01-17
3,Bought too much red cabbage? Turn it into a fe...,This fantastic vegan centrepiece makes full us...,['Tom Hunt'],"I devised today’s nut roast for Oddbox, a veg ...",2023-12-22
4,‘This year has been very good’: readers’ UK bu...,Readers share their favourite sightings over t...,['Guardian readers'],‘Constant companions to our gardening’A peacoc...,2023-12-19


In [58]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30059 entries, 0 to 30058
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Title           29111 non-null  str  
 1   Intro Text      29977 non-null  str  
 2   Authors         25489 non-null  str  
 3   Article Text    29691 non-null  str  
 4   Date Published  27618 non-null  str  
dtypes: str(5)
memory usage: 149.8 MB


In [59]:
df.drop(columns=['Authors', 'Title', 'Intro Text'], inplace=True)

In [60]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30059 entries, 0 to 30058
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Article Text    29691 non-null  str  
 1   Date Published  27618 non-null  str  
dtypes: str(2)
memory usage: 142.0 MB


In [61]:
df.head()

,Article Text,Date Published
0,Liz Truss will sign off on a push for more oil...,2022-08-30
1,It is an area so tranquil that the notion of b...,2021-03-22
2,"Visits to parks, community gardens and other u...",2023-01-17
3,"I devised today’s nut roast for Oddbox, a veg ...",2023-12-22
4,‘Constant companions to our gardening’A peacoc...,2023-12-19


In [62]:
df.dropna(inplace=True)

In [ ]:
patterns = {
    r"[a-zA-Z]+n\'t": 'not',        # 1. Contrazioni prima di tutto
    r'(http|www)[^\s]+': '',         # 2. Rimuovi URL
    r'\d+': '',                      # 3. Rimuovi numeri
    r'[^\w\s]': '',                  # 4. Rimuovi punteggiatura
    r'\b\w{1,2}\b': '',              # 5. Rimuovi token corti
    r'\s+': ' ',                     # 6. Normalizza spazi
}

def clean_column(df, column, patterns):
    for pattern, replacement in patterns.items():
        df[column] = df[column].str.replace(pattern, replacement, regex=True)
    df[column] = df[column].str.lower()
    return df

clean_column(df, 'Article Text', patterns)

,Article Text,Date Published
0,liz truss will sign off push for more oil dr...,2022-08-30
1,area tranquil that the notion bitter disp...,2021-03-22
2,visits parks community gardens and other urba...,2023-01-17
3,devised todays nut roast for oddbox veg box ...,2023-12-22
4,constant companions our gardeninga peacock bu...,2023-12-19
...,...,...
30054,insurers have warned that climate change could...,2020-11-12
30055,republican lawmaker has proposed that the in...,2021-06-05
30056,crossparty group federal mps has asked the a...,2018-04-25
30057,like millions their compatriots around the gl...,2020-01-06


In [ ]:
# --- TROPPE RIGHE CI METTEVA TROPPO TEMPO ---

# nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

# def lemmatizza_colonna(colonna):
#     lemmi_totali = []
#     testi = colonna.astype(str).tolist()
    
#     for doc in nlp.pipe(testi, batch_size=500, n_process=1):
#         # Unisce i lemmi di ogni parola separandoli con uno spazio
#         lemmi_testo = " ".join([token.lemma_ for token in doc])
#         lemmi_totali.append(lemmi_testo)
#     return lemmi_totali

# df["testo_lemmatizzato"] = lemmatizza_colonna(df["Article Text"])

In [64]:
def lemmatizza_colonna(colonna):
    testi = colonna.astype(str).tolist()
    lemmi_totali = []

    for testo in tqdm(testi, desc="Lemmatizzazione"):
        lemmi = ' '.join(simplemma.lemmatize(word, lang='en') for word in testo.split())
        lemmi_totali.append(lemmi)
    return lemmi_totali

df['testo_lemmatizzato'] = lemmatizza_colonna(df['Article Text'])


Lemmatizzazione: 100%|██████████| 27565/27565 [00:25<00:00, 1101.61it/s]


In [65]:
df.head()

,Article Text,Date Published,testo_lemmatizzato
0,liz truss will sign off push for more oil dr...,2022-08-30,liz truss will sign off push for more oil dril...
1,area tranquil that the notion bitter disp...,2021-03-22,area tranquil that the notion bitter dispute h...
2,visits parks community gardens and other urba...,2023-01-17,visit park community garden and other urban gr...
3,devised todays nut roast for oddbox veg box ...,2023-12-22,devise today nut roast for oddbox veg box outf...
4,constant companions our gardeninga peacock bu...,2023-12-19,constant companion our gardeninga peacock butt...
